# 01 — Build and test (CPU runtime)

**Runtime > Change runtime type > CPU**.

Run `00_setup.ipynb` first in this session (or re-run its clone/apt cells). This notebook configures Release CMake, builds, runs `ctest`, then a ThreadSanitizer build.

In [ ]:
import os
import subprocess

REPO_URL = "https://github.com/ShubhamDudhbhate-VJTI/Scientific_Calculator.git"
CLONE_DIR = "/content/parallel-sci-calc"
DRIVE_ROOT = "/content/drive/MyDrive/parallel-sci-calc"

# Colab wipes /content when you start a new runtime. Clone again if needed.
if not os.path.isfile(os.path.join(CLONE_DIR, "CMakeLists.txt")):
    print("Repo not in this runtime. Cloning", REPO_URL)
    subprocess.check_call(["git", "clone", REPO_URL, CLONE_DIR])
else:
    subprocess.call(["git", "-C", CLONE_DIR, "pull", "--ff-only"])

assert os.path.isfile(os.path.join(CLONE_DIR, "CMakeLists.txt")), "FAIL: clone did not produce CMakeLists.txt"
os.makedirs(os.path.join(DRIVE_ROOT, "logs"), exist_ok=True)
os.makedirs(os.path.join(DRIVE_ROOT, "results"), exist_ok=True)
print("OK: repo at", CLONE_DIR)

In [ ]:
from google.colab import drive
import os, subprocess, shutil

if not os.path.isdir("/content/drive/MyDrive"):
    drive.mount("/content/drive")
print("OK: Drive ready")

if shutil.which("cmake") is None or shutil.which("g++") is None:
    subprocess.check_call(["apt-get", "update", "-qq"])
    subprocess.check_call(["apt-get", "install", "-y", "-qq", "cmake", "g++", "libomp-dev", "git"])
print("OK: cmake/g++ present")

In [ ]:
%%bash
set -euo pipefail
cd /content/parallel-sci-calc
git pull --ff-only || true
chmod +x scripts/*.sh
bash scripts/colab_build.sh --jobs "$(nproc)" 2>&1 | tee /content/drive/MyDrive/parallel-sci-calc/logs/build_release.txt
echo "OK: Release build and ctest finished (see log if the command failed)"

In [ ]:
%%bash
set -euo pipefail
cd /content/parallel-sci-calc
./build/psc_machine_info --out /content/drive/MyDrive/parallel-sci-calc/results/machine_info_from_binary.txt
echo "OK: psc_machine_info"

In [ ]:
%%bash
set -euo pipefail
cd /content/parallel-sci-calc
echo "help" | ./build/psc | head -n 20
./build/psc --expr "3 + 4 * (2 - 1)^2"
./build/psc --expr "-2^2"
echo "OK: REPL smoke"

In [ ]:
%%bash
set -euo pipefail
cd /content/parallel-sci-calc
echo "--- ThreadSanitizer build (OpenMP+TSan may fail on some GCC/libgomp; paste the log if so) ---"
if bash scripts/colab_build.sh --tsan --jobs "$(nproc)" 2>&1 | tee /content/drive/MyDrive/parallel-sci-calc/logs/build_tsan.txt; then
  echo "OK: ThreadSanitizer ctest finished"
else
  echo "FAIL: ThreadSanitizer build or tests. Copy build_tsan.txt back to Cursor. Do not claim TSan is clean."
  exit 1
fi